# Protocolo de validación live ↔ backtest — STRATA / M10

**Propósito.** Documentar el protocolo de 6 capas que garantiza que los resultados observados en backtest se corresponderán con los del despliegue operativo. Este notebook **NO ejecuta** ninguna validación: documenta la metodología y el código que la implementaría.

**Decisión metodológica.** Antes de desplegar M10-v6 (o cualquier variante M10) en `live/daily_run.py`, se requiere que **al menos 4 de las 6 capas** se hayan ejecutado con éxito. Si alguna capa critica falla, el modo live se queda con M8 (regla a mano interpretable).

**Trazabilidad.**
- BITACORA 2026-06-15 — entradas M10-v2 → v6 (iteraciones del meta-learner).
- `STRATA_kit/INVESTIGACION_VALIDACION_TIEMPO_REAL.md` (investigación bibliográfica previa).
- `experiments/m10_v6_walkforward_cpcv_intra.py` — código operativo deployable.
- `live/daily_run.py` — script live actual (mantiene M8 por defecto).

**Tabla de contenidos.**

| # | Capa | Propósito | Coste estimado |
|---|---|---|---|
| §1 | Multi-seed robustness | Estabilidad ante semilla | ~5 min |
| §2 | Multi-origin walk-forward | Estabilidad ante punto inicio OOS | ~10 min |
| §3 | Out-of-time holdout | Generalización al futuro inmediato | ~3 min |
| §4 | Drift monitoring en live | Alertas automáticas en producción | infraestructura permanente |
| §5 | Paper trading 30 días | Equivalencia empírica live ↔ backtest | 30 días + 5 min análisis |
| §6 | A/B testing M8 vs M10-v6 | Comparación operativa pareada | 90+ días en paralelo |

---

## §1 — Multi-seed robustness

**Pregunta de validación.** ¿El resultado de M10-v6 (Sharpe +1.64, equity €1044) depende de la semilla aleatoria del XGBoost?

**Hipótesis.** σ(Sharpe across 5 seeds) < 0.20 ⇒ el resultado es estadísticamente estable y la mejora vs M8 (Sharpe +0.66) no es artefacto de una sola semilla afortunada.

**Justificación teórica.** Demšar 2006 *Statistical Comparisons of Classifiers over Multiple Data Sets* establece que cualquier resultado de un clasificador estocástico debe reportarse con desviación típica sobre al menos 5 inicializaciones distintas.

**Coste computacional.** 5 × (55 refits × 14 modelos) ≈ 50 segundos. Despreciable.

**Criterio de éxito pre-fijado.**
- σ(Sharpe) < 0.20
- σ(equity_final) < 0.02 (€20 sobre €1000)
- Sharpe medio sobre 5 seeds > +1.0
- **Si falla:** M10-v6 es inestable ante semilla; no deployable hasta entenderlo.

In [ ]:
# CÓDIGO DOCUMENTADO — NO EJECUTAR
# Para ejecutar de verdad, descomentar y correr con --execute en jupyter nbconvert

from experiments.m10_v6_walkforward_cpcv_intra import run_m10_v6_cpcv_intra
import pandas as pd
import numpy as np

SEEDS = [42, 123, 456, 789, 2024]  # pre-fijados, no se prueban más

results = []
for seed in SEEDS:
    payload = run_m10_v6_cpcv_intra(
        ticker="SPY",
        end_date="2026-06-02",
        seed=seed,
    )
    results.append({
        "seed": seed,
        "sharpe": payload["metrics"]["sharpe"],
        "equity_final": payload["equity_final"],
        "max_drawdown": payload["metrics"]["max_drawdown"],
        "logloss_cal": payload["logloss_cal_oof_final"],
    })

df_seeds = pd.DataFrame(results)
print(df_seeds.to_string(index=False))
print(f"\nσ(Sharpe) = {df_seeds['sharpe'].std():.3f}")
print(f"σ(equity) = {df_seeds['equity_final'].std():.4f}")
print(f"Sharpe medio: {df_seeds['sharpe'].mean():.3f}")

# Veredicto
passes = (df_seeds['sharpe'].std() < 0.20 and 
          df_seeds['equity_final'].std() < 0.02 and 
          df_seeds['sharpe'].mean() > 1.0)
print(f"\nCapa §1: {'✅ PASA' if passes else '❌ FALLA'}")

**Output esperado (basado en M10-v6 canónico).**

```
  seed  sharpe  equity_final  max_drawdown  logloss_cal
    42   1.638        1.0440        -0.009        0.707
   123   1.5xx        1.04xx        -0.01x        0.7xx
   456   1.7xx        1.04xx        -0.01x        0.7xx
   789   1.6xx        1.04xx        -0.01x        0.7xx
  2024   1.6xx        1.04xx        -0.01x        0.7xx

σ(Sharpe) ≈ 0.10-0.15  (esperado)
σ(equity) ≈ 0.005      (esperado)
```

---

## §2 — Multi-origin walk-forward

**Pregunta de validación.** ¿El resultado depende del punto exacto de inicio del OOS?

**Hipótesis.** El Sharpe y equity son robustos ante desplazar el inicio del OOS hasta 2 meses.

**Justificación teórica.** Bergmeir & Hyndman 2018 sobre rolling origin: la elección arbitraria del start point puede sesgar resultados. Probar 3 puntos (un mes de diferencia entre ellos) cubre la sensibilidad temporal sin reducir excesivamente el OOS efectivo.

**Coste computacional.** 3 × 50 seg ≈ 2.5 minutos.

**Criterio de éxito pre-fijado.**
- |Δ equity entre orígenes| / equity_baseline < 0.05
- Sharpe positivo en los 3 orígenes
- **Si falla:** el resultado depende crucialmente del punto de inicio elegido, lo que indica overfitting al primer mes del OOS.

In [ ]:
# CÓDIGO DOCUMENTADO — NO EJECUTAR
# NOTA: esto requiere modificar _build_dataset para aceptar oos_start opcional.
# Ver implementación propuesta en experiments/m10_v6_multi_origin.py (no creado aún).

ORIGINS = ["2024-10-01", "2024-11-01", "2024-12-01"]  # pre-fijados

results = []
for start in ORIGINS:
    payload = run_m10_v6_with_oos_start(  # función a implementar
        ticker="SPY",
        oos_start=start,
        end_date="2026-06-02",
        seed=42,
    )
    results.append({
        "start": start,
        "n_obs": payload["n_obs"],
        "sharpe": payload["metrics"]["sharpe"],
        "equity_final": payload["equity_final"],
    })

df_origins = pd.DataFrame(results)
print(df_origins.to_string(index=False))

baseline_equity = df_origins.iloc[0]["equity_final"]
max_dev = max(abs(df_origins["equity_final"] - baseline_equity)) / baseline_equity
print(f"\nMax desviación relativa: {max_dev:.4f}")
print(f"Capa §2: {'✅ PASA' if max_dev < 0.05 and (df_origins['sharpe'] > 0).all() else '❌ FALLA'}")

---

## §3 — Out-of-time holdout

**Pregunta de validación.** Si reservamos los últimos 60 días del OOS como holdout 100% no visto (no usados ni para entrenar XGBoost, ni para calibrar Platt, ni para calcular percentiles de abstención/P95), ¿el rendimiento en el holdout es similar al del periodo de train?

**Hipótesis.** log-loss_holdout ≤ log-loss_train × 1.05 ⇒ generalización aceptable al futuro inmediato.

**Justificación teórica.** El holdout no contaminado es el estándar académico en evaluación de modelos predictivos (Hastie-Tibshirani-Friedman 2009, cap 7.10). En este caso es especialmente relevante porque verifica que el modelo no se beneficia de información del periodo completo del OOS vía la calibración isotónica/Platt.

**Coste computacional.** 1 × 50 seg + análisis ≈ 3 minutos.

**Criterio de éxito pre-fijado.**
- log-loss_holdout < log(2) × 1.05 (señal aprendible en los últimos 60 días)
- accuracy_holdout > 0.50 sobre días activos
- Sharpe_holdout > 0 (no destruye valor en el periodo no visto)
- **Si falla:** el modelo no generaliza al futuro inmediato; alta probabilidad de fallo en live.

In [ ]:
# CÓDIGO DOCUMENTADO — NO EJECUTAR

HOLDOUT_DAYS = 60  # últimos 60 días bursátiles del OOS

from experiments.m10_ml_meta import _build_dataset
X, y, sigma_oos, returns_oos = _build_dataset("SPY", "2026-06-02", None)

# Split: train = todo menos últimos 60 días, holdout = últimos 60
split_idx = len(X) - HOLDOUT_DAYS
X_train, X_holdout = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_holdout = y.iloc[:split_idx], y.iloc[split_idx:]

# Entrenar M10-v6 SOLO sobre train
from experiments.m10_v6_walkforward_cpcv_intra import fit_cpcv_intra_ensemble
from experiments.m10_v4_walkforward import platt_fit_transform

ensemble = fit_cpcv_intra_ensemble(
    X_train.values.astype(np.float32),
    y_train.values.astype(int),
    seed=42,
)

# Predecir en holdout
p1_holdout_raw = np.array([
    np.mean([m.predict_proba(X_holdout.iloc[[i]].values.astype(np.float32))[0, 1] 
             for m in ensemble])
    for i in range(len(X_holdout))
])

# Log-loss en holdout vs train
eps = 1e-7
ll_holdout = -np.mean(
    y_holdout.values * np.log(np.clip(p1_holdout_raw, eps, 1-eps)) +
    (1 - y_holdout.values) * np.log(np.clip(1 - p1_holdout_raw, eps, 1-eps))
)
print(f"Log-loss holdout (últimos {HOLDOUT_DAYS} días): {ll_holdout:.4f}")
print(f"Baseline log(2): {np.log(2):.4f}")
print(f"Capa §3: {'✅ PASA' if ll_holdout < np.log(2) * 1.05 else '❌ FALLA'}")

---

## §4 — Drift monitoring en live (infraestructura)

**Pregunta de validación.** Una vez M10-v6 esté en live, ¿cómo detectamos que el modelo se está desviando del comportamiento de backtest?

**Justificación teórica.** Lu et al. 2018 *Learning under Concept Drift: A Review* clasifica drift en abrupto, gradual, incremental y recurrente. Las métricas rolling sobre ventana de 20 días detectan los cuatro tipos con falsos positivos aceptables.

**Cuatro métricas rolling (ventana 20 días bursátiles).**

| Métrica | Definición | Detecta |
|---|---|---|
| `rolling_logloss` | mean log-loss últimos 20 días | calidad de calibración |
| `rolling_accuracy` | hit rate direccional | calidad direccional |
| `rolling_p1_mean` | media de p1 calibrado | sesgo sistemático |
| `rolling_abst_rate` | fracción de días con direction=0 | cambio en confianza |

**Reglas de alerta pre-fijadas.**

| Alerta | Condición | Acción |
|---|---|---|
| **Verde** | todas las métricas en banda backtest ± 1σ | continuar |
| **Amarilla** | 1 métrica fuera de banda ± 2σ por 3 días | revisar manualmente |
| **Naranja** | 2 métricas fuera de banda ± 2σ por 3 días | reducir exposición 50% |
| **Roja** | `rolling_logloss > backtest_logloss + 2σ` por 5 días consecutivos | **pausar M10, volver a M8** |

**Decisión operativa.** Cualquier alerta roja → switch automático a M8 hasta investigación manual.

**Coste computacional.** Despreciable (4 medias móviles por día).

In [ ]:
# CÓDIGO DOCUMENTADO — diseño del módulo live/drift_monitor.py

from dataclasses import dataclass
from typing import Literal
import numpy as np
import pandas as pd

AlertLevel = Literal["green", "yellow", "orange", "red"]


@dataclass
class BacktestBaseline:
    """Valores de referencia del backtest M10-v6 canónico."""
    logloss_mean: float = 0.707
    logloss_std: float = 0.05  # estimar de fold variability del backtest
    accuracy_mean: float = 0.604
    accuracy_std: float = 0.05
    p1_mean: float = 0.557
    p1_std: float = 0.10
    abst_rate_mean: float = 0.244
    abst_rate_std: float = 0.05


def compute_rolling_metrics(
    live_log: pd.DataFrame,
    window: int = 20,
) -> pd.DataFrame:
    """Calcula métricas rolling sobre los últimos `window` días de live."""
    # live_log columns: date, p1_cal, direction, y_realized
    eps = 1e-7
    live_log = live_log.copy()
    p_clip = live_log["p1_cal"].clip(eps, 1-eps)
    live_log["logloss_day"] = -(
        live_log["y_realized"] * np.log(p_clip) +
        (1 - live_log["y_realized"]) * np.log(1 - p_clip)
    )
    live_log["hit"] = (np.sign(live_log["direction"]) == np.sign(
        2*live_log["y_realized"] - 1)).astype(int)
    live_log["abstained"] = (np.abs(live_log["direction"]) < 1e-9).astype(int)
    
    rolling = pd.DataFrame({
        "rolling_logloss": live_log["logloss_day"].rolling(window).mean(),
        "rolling_accuracy": live_log["hit"].rolling(window).mean(),
        "rolling_p1_mean": live_log["p1_cal"].rolling(window).mean(),
        "rolling_abst_rate": live_log["abstained"].rolling(window).mean(),
    })
    return rolling


def evaluate_alert(
    metrics: dict,
    baseline: BacktestBaseline,
    days_above_threshold: int,
) -> AlertLevel:
    """Decide nivel de alerta según las 4 reglas pre-fijadas."""
    out_of_band = 0
    for key, val in metrics.items():
        ref = getattr(baseline, key.replace("rolling_", "") + "_mean")
        std = getattr(baseline, key.replace("rolling_", "") + "_std")
        if abs(val - ref) > 2 * std:
            out_of_band += 1
    
    # Regla roja: logloss > backtest + 2σ por 5 días consecutivos
    if (metrics["rolling_logloss"] > baseline.logloss_mean + 2 * baseline.logloss_std
            and days_above_threshold >= 5):
        return "red"  # → switch automático a M8
    if out_of_band >= 2 and days_above_threshold >= 3:
        return "orange"
    if out_of_band == 1 and days_above_threshold >= 3:
        return "yellow"
    return "green"

**Integración en `live/daily_run.py`.**

```python
# Después de calcular position
rolling = compute_rolling_metrics(load_live_log(), window=20)
metrics_today = rolling.iloc[-1].to_dict()
alert = evaluate_alert(metrics_today, BacktestBaseline(), days_above_threshold=...)

if alert == "red":
    print("⚠️ Alerta roja — switch a M8")
    position = compute_m8(today)  # fallback
    log_event("drift_red_switch_to_m8")

log_alert(alert, metrics_today)
```

---

## §5 — Paper trading 30 días

**Pregunta de validación.** Si ejecutamos M10-v6 en live durante 30 días sin invertir dinero real, ¿las posiciones generadas día a día coinciden con las que produciría el backtest sobre el mismo periodo cuando ese periodo se vuelva 'pasado'?

**Hipótesis.** |position_live(t) − position_backtest(t)| < 0.01 en >95% de los días.

**Justificación teórica.** La equivalencia live↔backtest es la condición operativa central que López de Prado (2018, cap. 7.4.3) reclama como requisito para deployment. Es la única forma de verificar que (a) no hay leakage operativo accidental, (b) la calibración acumulada se actualiza correctamente, (c) los percentiles de abstención y P95 son consistentes.

**Protocolo paso a paso.**

1. **Día 0:** snapshot del modelo M10-v6 entrenado con datos hasta hoy.
2. **Días 1-30 (en live):** cada día ejecutar `daily_run.py` + log de `(p1_raw_live, p1_cal_live, direction_live, position_live, ensemble_hash)`.
3. **Día 31:** re-ejecutar M10-v6 en backtest sobre los días 1-30, con datos hasta el día 30.
4. **Comparar día a día:** posición live vs posición backtest.

**Criterio de equivalencia.**
- |position_live − position_backtest| < 0.01 en ≥95% de los 30 días.
- Hash del ensemble coincide en cada lunes (refit semanal idéntico).
- Sharpe(live 30d) y Sharpe(backtest 30d) dentro de ±0.5.

**Coste computacional.** 30 días calendario + 5 minutos análisis.

In [ ]:
# CÓDIGO DOCUMENTADO — verificación paper trading

from pathlib import Path
import json
import pandas as pd
import numpy as np

# 1. Cargar log de live
live_records = []
for f in sorted(Path("outputs/live/m10_v6_paper/").glob("*.json")):
    d = json.loads(f.read_text())
    live_records.append({
        "date": d["date"],
        "position_live": d["m10_v6"]["position"],
        "p1_cal_live": d["m10_v6"]["p1_cal"],
        "ensemble_hash_live": d["m10_v6"]["ensemble_hash"],
    })
df_live = pd.DataFrame(live_records).set_index("date")

# 2. Re-ejecutar M10-v6 en backtest sobre el mismo periodo
from experiments.m10_v6_walkforward_cpcv_intra import run_m10_v6_cpcv_intra
backtest_end = df_live.index[-1]
backtest_payload = run_m10_v6_cpcv_intra(
    ticker="SPY",
    end_date=backtest_end,
    seed=42,
)
weights_bt = pd.Series(
    [float(v) for v in backtest_payload["weights"].values()],
    index=pd.to_datetime(list(backtest_payload["weights"].keys())),
).reindex(pd.to_datetime(df_live.index))

# 3. Comparar
df_live["position_backtest"] = weights_bt.values
df_live["abs_diff"] = abs(df_live["position_live"] - df_live["position_backtest"])
pct_within_tolerance = (df_live["abs_diff"] < 0.01).mean()

print(f"Días con |Δ position| < 0.01: {pct_within_tolerance:.1%}")
print(f"Max diff: {df_live['abs_diff'].max():.4f}")
print(f"Capa §5: {'✅ PASA' if pct_within_tolerance >= 0.95 else '❌ FALLA'}")

---

## §6 — A/B testing M8 vs M10-v6 en paralelo

**Pregunta de validación.** Comparación pareada operativa real (no backtest) — ¿M10-v6 supera a M8 en condiciones de despliegue?

**Hipótesis.** Durante 90+ días, los retornos diarios de M10-v6 y M8 satisfacen:
- Diebold-Mariano p < 0.05 a favor de M10-v6 → M10-v6 se convierte en default.
- DM p > 0.05 → empate operativo, M8 mantiene default por interpretabilidad.

**Justificación teórica.** Diebold-Mariano 1995 sobre comparación de forecasts pareados; Wilcoxon 1945 para robustez no-paramétrica; Politis-Romano 1994 (bootstrap estacionario) para IC95% de Δ Sharpe.

**Protocolo.** Ambas estrategias ejecutándose en paralelo, mismo agente, mismo HMM/GARCH. Solo cambia la regla de sizing final.

**Coste computacional.** 90+ días calendario + análisis estadístico.

In [ ]:
# CÓDIGO DOCUMENTADO — análisis A/B M8 vs M10-v6

from scipy.stats import wilcoxon, norm
from core.stats import stationary_bootstrap_ci
import pandas as pd
import numpy as np

# Cargar live log (asumiendo que cada día guarda PnL de ambas estrategias)
ab_records = []
for f in sorted(Path("outputs/live/ab_test/").glob("*.json")):
    d = json.loads(f.read_text())
    ab_records.append({
        "date": d["date"],
        "pnl_m8": d["m8"]["pnl_realized"],
        "pnl_m10": d["m10_v6"]["pnl_realized"],
    })
df_ab = pd.DataFrame(ab_records).set_index("date")

# Diebold-Mariano (asintóticamente normal sobre diferencia de PnL)
d = df_ab["pnl_m10"] - df_ab["pnl_m8"]
t_stat = d.mean() / np.sqrt(d.var(ddof=1) / len(d))
p_dm = 2 * (1 - norm.cdf(abs(t_stat)))
print(f"Diebold-Mariano: t={t_stat:+.3f}, p={p_dm:.4f}")

# Wilcoxon signed-rank (robusto a outliers)
w_stat, p_wilcoxon = wilcoxon(df_ab["pnl_m10"], df_ab["pnl_m8"])
print(f"Wilcoxon: W={w_stat:.0f}, p={p_wilcoxon:.4f}")

# Bootstrap estacionario IC95% Δ Sharpe (Politis-Romano)
diff_returns = (df_ab["pnl_m10"] - df_ab["pnl_m8"]).values
low, high, point = stationary_bootstrap_ci(
    diff_returns,
    statistic=lambda x: x.mean() / x.std() * np.sqrt(252) if x.std() > 0 else 0,
    alpha=0.05,
)
print(f"IC95% Δ Sharpe (Politis-Romano): [{low:+.3f}, {high:+.3f}], punto={point:+.3f}")

# Decisión operativa
if p_dm < 0.05 and t_stat > 0:
    decision = "✅ M10-v6 vuelve a default en live"
elif p_dm > 0.05:
    decision = "⚠️ Empate operativo. M8 mantiene default por interpretabilidad."
else:
    decision = "❌ M10-v6 peor que M8 significativamente. Mantener M8 y revisar."
print(f"\nDecisión: {decision}")

---

## §A — Limitaciones honestas documentadas

Para reportar en la memoria del TFG independientemente del resultado de las 6 capas:

1. **Sample size pequeño:** N ≈ 400 días en el OOS unificado (2024-10 → 2026-06). Cualquier conclusión está condicionada a este tamaño muestral. El régimen de muestra pequeña documentado en Bergmeir-Hyndman 2018 implica que el gap CPCV ↔ walk-forward es estructural y puede converger asintóticamente solo con N → ∞.

2. **Mercado bull en OOS:** B&H S&P 500 +29.5% sobre el OOS. Sesgo de muestra hacia régimen alcista. STRATA → diseñada como disciplina de riesgo, no captura plenamente rallies pasivos.

3. **Features débiles:** Top 5 SHAP de M10 son las 3 features STRATA + 2 régimen. Ninguna personalidad del agente entra al top 5. Esto sugiere que la señal aprovechable es limitada — incluso un meta-learner disciplinado solo logra log-loss cercano a log(2) bajo CPCV completo.

4. **HMM-SPY como proxy macro:** sobre el panel multi-activo no-SPY, M10 ingenuo destruye valor en 6/9 tickers (M10 < M5). M10-v3 disciplinado mejora pero no resuelve la transferibilidad. Para deployment cross-asset habría que calibrar HMM por ticker — no realizado.

5. **Cutoff temporal DeepSeek V3:** el OOS empieza 2024-10-01, posterior al cutoff conocido del LLM. Si el modelo se actualiza en el futuro (cutoff posterior), el sesgo se contaminaría. Considerar re-evaluar con cutoff actualizado al desplegar.

6. **Dependencia de calibración HMM/GARCH/BOCPD:** los modelos clásicos están calibrados sobre 2000-2024-09 (24 años) y no se re-calibran en live. Si las propiedades estadísticas del mercado cambian materialmente, los percentiles RAM/PSA/GSO podrían quedar obsoletos. Re-calibración anual recomendada.

---

## §B — Costes computacionales totales

| Capa | Coste único | Coste continuo (en live) |
|---|---|---|
| §1 Multi-seed | 5 min | — |
| §2 Multi-origin | 10 min | — |
| §3 Holdout | 3 min | — |
| §4 Drift monitoring | ~30 min setup | <1 segundo/día |
| §5 Paper trading | 5 min setup + 30 días calendario | 45 seg/lunes + 0.1 seg/día |
| §6 A/B testing | 5 min setup + 90 días calendario | igual a §5 |

**Coste único agregado:** ~1 hora de setup técnico.
**Coste continuo en producción:** despreciable (<2 minutos/semana).

---

## §C — Frase de defensa final para el tribunal

> *"El TFG documenta un protocolo de validación live ↔ backtest de 6 capas: multi-seed robustness (Demšar 2006), multi-origin walk-forward (Bergmeir-Hyndman 2018), out-of-time holdout (Hastie 2009), drift monitoring con alertas (Lu et al. 2018), paper trading 30 días (López de Prado 2018), A/B testing pareado con Diebold-Mariano (1995) y bootstrap estacionario (Politis-Romano 1994). El despliegue de M10-v6 en producción está condicionado a que ≥4 de las 6 capas se ejecuten con éxito; mientras tanto el modo live mantiene M8 (regla a mano interpretable) como configuración por defecto. Esta separación blinda contra el riesgo de drift no detectado y respeta la separación recomendada por López de Prado (2018, cap. 11) entre evaluación retrospectiva y despliegue operativo."*

---

## Trazabilidad final

- BITACORA pre-registros M10-v2 → v7 (2026-06-15) — todas las iteraciones honestas.
- `STRATA_kit/INVESTIGACION_VALIDACION_TIEMPO_REAL.md` — investigación bibliográfica previa.
- `STRATA_kit/M10_V7_GUIA.md` — guía técnica de Kelly + regime tilt (cuando se cree).
- `experiments/m10_v6_walkforward_cpcv_intra.py` — código deployable.
- `experiments/m10_v7_kelly_regime_tilt.py` — variantes v7a-v7d (cuando se ejecuten).
- `core/stats.py:deflated_sharpe` — Bailey-López de Prado 2014.
- `live/daily_run.py` — modo live actual (M8).